# Writing your own Neural Network code

In [1]:
import autograd.numpy as np
import pandas as pd

# custom imports
from runge_preprocessing import x, x_train, x_train_scaled, x_test, y, y_noise, y_train, y_test, layer_dim, activations, activations_derivative, ETA_VALUES, LAMBDA_VALUES, MOMENTUM, RUNGE_MAX_ITERATIONS, VERBOSE
from neural_network import NN
import schedulers


TypeError: sigmoid() missing 1 required positional argument: 'Z'

In [ ]:



import time
from cost_functions import mse
mse_formula = mse().cost

runge_NN = NN(dims = layer_dim, activation_funcs = activations, activation_ders = activations_derivative)

def neural_network_loop(etas, lambdas, optimizer_name, max_iterations, momentum_val=0.9, verbose=True):


    results = []

    for eta in etas:
        for lmbd in lambdas:
            
            if verbose:
                print(f"\nTraining with: optimizer={optimizer_name}, lr={eta}, lambda={lmbd}, iteration={max_iterations}")

            start_time = time.time()

            if optimizer_name == 'ADAM':
                optimizer = schedulers.ADAM(eta, rho=lmbd, rho2=0)  
            if optimizer_name == 'ADAM_L1':
                optimizer = schedulers.ADAM(eta, rho=lmbd, rho2=0)  
            if optimizer_name == 'ADAM_L2':
                optimizer = schedulers.ADAM(eta, rho=0, rho2=lmbd)   
            elif optimizer_name == 'SGD':
                optimizer = schedulers.momentum(eta, momentum=momentum_val)
            elif optimizer_name == 'RMSprop':
                optimizer = schedulers.RMSprop(eta=etas,rho=lmbd)

            epoch_scores, predictions = runge_NN.fit(X=x_train_scaled, 
                                                     t=y_train, 
                                                     X_val=x_test, 
                                                     t_val=y_test, 
                                                     epochs=max_iterations, 
                                                     scheduler=optimizer)
            
            final_mse = mse_formula(y_true=y_test, y_pred=predictions)
            
            elapsed_time = time.time() - start_time

            results.append({
                'Learning Rate': eta,
                'Lambda': lmbd,
                'Iterations': max_iterations,
                'Elapsed Time (s)': round(elapsed_time, 2),
                'Training Errors': epoch_scores['training_errors'],
                'Validation Errors': epoch_scores['validation_errors'],
                'MSE': final_mse,
                'Predictions': predictions
            })
    return pd.DataFrame(results)

results_sgd = neural_network_loop(ETA_VALUES, LAMBDA_VALUES, 'SGD', RUNGE_MAX_ITERATIONS, momentum_val=MOMENTUM, verbose=VERBOSE)
results_rmsprop = neural_network_loop(ETA_VALUES, LAMBDA_VALUES, 'RMSprop', RUNGE_MAX_ITERATIONS, momentum_val=MOMENTUM, verbose=VERBOSE)
results_adam_no_penalty = neural_network_loop(ETA_VALUES, LAMBDA_VALUES, 'ADAM', RUNGE_MAX_ITERATIONS, momentum_val=MOMENTUM, verbose=VERBOSE) 
results_adam_L1 = neural_network_loop(ETA_VALUES, LAMBDA_VALUES, 'ADAM_L1', RUNGE_MAX_ITERATIONS, momentum_val=MOMENTUM, verbose=VERBOSE) 
results_adam_L2 = neural_network_loop(ETA_VALUES, LAMBDA_VALUES, 'ADAM_L2', RUNGE_MAX_ITERATIONS, momentum_val=MOMENTUM, verbose=VERBOSE) 



In [ ]:
from plotting import plot_heatmap

SHOW_PLOT = True

plot_heatmap(results_sgd, title='Runge function with SGD', heat_metric='MSE', filename=f'runge_heatmap_sgd_mse_iter{RUNGE_MAX_ITERATIONS}_momentum{MOMENTUM}', show_plot=SHOW_PLOT)
plot_heatmap(results_rmsprop, title='Runge function with RMSprop', heat_metric='MSE', filename=f'runge_heatmap_rmsprop_mse_iter{RUNGE_MAX_ITERATIONS}_momentum{MOMENTUM}', show_plot=SHOW_PLOT)
plot_heatmap(results_adam_no_penalty, title='Runge function with ADAM - no penalty', heat_metric='MSE', filename=f'runge_heatmap_adamNO_mse_iter{RUNGE_MAX_ITERATIONS}_momentum{MOMENTUM}', show_plot=SHOW_PLOT)
plot_heatmap(results_adam_L1, title='Runge function with ADAM - L1', heat_metric='MSE', filename=f'runge_heatmap_adamL1_mse_iter{RUNGE_MAX_ITERATIONS}_momentum{MOMENTUM}', show_plot=SHOW_PLOT)
plot_heatmap(results_adam_L2, title='Runge function with ADAM - L2', heat_metric='MSE', filename=f'runge_heatmap_adamL2_mse_iter{RUNGE_MAX_ITERATIONS}_momentum{MOMENTUM}', show_plot=SHOW_PLOT)


In [ ]:
from plotting import plot_runges

SHOW_PLOT = True

# SGD
min_row = results_sgd.loc[results_sgd['MSE'].idxmin()]
predictions = min_row['Predictions'] # find predictions for lowest mse
plot_runges(x, y, x_test, predictions, title=f'Runge function - SGD', filename=f'runge_predicted_iter{RUNGE_MAX_ITERATIONS}_momentum{MOMENTUM}', show_plot=SHOW_PLOT)

# RMSprop
min_row = results_rmsprop.loc[results_sgd['MSE'].idxmin()]
predictions = min_row['Predictions'] # find predictions for lowest mse
plot_runges(x, y, x_test, predictions, title=f'Runge function - RMSprop', filename=f'runge_predicted_rmsprop_iter{RUNGE_MAX_ITERATIONS}_momentum{MOMENTUM}', show_plot=SHOW_PLOT)

# ADAM - no penalty
min_row = results_adam_no_penalty.loc[results_sgd['MSE'].idxmin()]
predictions = min_row['Predictions'] # find predictions for lowest mse
plot_runges(x, y, x_test, predictions, title=f'Runge function - ADAM - no penalty', filename=f'runge_predicted_adamNO_iter{RUNGE_MAX_ITERATIONS}_momentum{MOMENTUM}', show_plot=SHOW_PLOT)

# ADAM - L1
min_row = results_adam_L1.loc[results_sgd['MSE'].idxmin()]
predictions = min_row['Predictions'] # find predictions for lowest mse
plot_runges(x, y, x_test, predictions, title=f'Runge function - ADAM - L1', filename=f'runge_predicted_adamL1_iter{RUNGE_MAX_ITERATIONS}_momentum{MOMENTUM}', show_plot=SHOW_PLOT)

# ADAM - L2
min_row = results_adam_L2.loc[results_sgd['MSE'].idxmin()]
predictions = min_row['Predictions'] # find predictions for lowest mse
plot_runges(x, y, x_test, predictions, title=f'Runge function - ADAM - L2', filename=f'runge_predicted_adamL2_iter{RUNGE_MAX_ITERATIONS}_momentum{MOMENTUM}', show_plot=SHOW_PLOT)